# Semana 9: RQ4, medidas de discordância no Rio Grande do Sul

Seção 3 de `prereg/plano_rq4.md`. **Este notebook não roda nenhum modelo e não precisa de GPU.** Ele lê as margens que o notebook de inferência gravou (`resultados/semana9/rq4_margens`), o τ de cada uma das 15 execuções (`resultados.json` da análise confirmatória) e as máscaras de referência, e calcula, para cada execução, cada tile, os três tiles juntos, cada distância da camada da UFRGS (1 km principal, 500 m e 2 km) e cada grupo:

| Medida | Definição |
|---|---|
| `taxa_discordancia` | P(previsão ≠ consenso das referências) |
| `auroc_d` | AUROC_e de `nucleo.py` com discordância no lugar de erro |
| `discordancia_nao_sinalizada` | P(MSP ≥ τ \| discordância) |
| `conjunto_vazio` | P(MSP < τ) |
| `discordancia_entre_sinalizados`, `discordancia_entre_nao_sinalizados` | taxa de discordância entre os pixels com MSP < τ e entre os com MSP ≥ τ |

Os grupos são todos os pixels, cada estrato do WorldCover 2021 (construído, água permanente, outro) e o número de referências que cobrem o pixel (1 ou 2). Em cada regime a tabela traz a média, o desvio e a faixa das 5 sementes. Para os pixels de todos os grupos juntos, ela traz também o valor do mesmo regime no alvo agregado da análise confirmatória e a faixa dos cinco eventos do alvo (`regimes.csv`), com a posição da média em relação a essa faixa (dentro, abaixo, acima). A posição é só descrita, nenhum teste é feito.

**Palavra.** As referências são dois mapas independentes da cheia, não são verdade de campo. Por isso a tabela diz sempre discordância, e na comparação com o alvo as colunas `taxa_erro`, `auroc`, `erro_nao_sinalizado` e `conjunto_vazio` da análise confirmatória são lidas como o mesmo cálculo com outro rótulo de referência.

Os pixels contados são os pontuados (código 0 ou 1) nas janelas que entram na distância considerada. O AUROC usa os histogramas de |m| de `nucleo.py` e é conferido, em cada execução e cada tile na distância principal, contra o valor exato; se a diferença passar de 1e-4, o notebook refina os intervalos pela regra da seção 6 do pré-registro e recalcula.

| Célula | O que faz |
|---|---|
| 1 | Monta o Drive, clona o repositório, lê os registros e a análise confirmatória |
| 2 | Lê os rótulos dos três tiles e o τ das 15 execuções, com as conferências |
| 3 | Calcula as medidas (20 a 40 min), grava as tabelas |
| 4 | Mapas de MSP, sinalizados e discordância (semente 1 de cada regime) |
| 5 | Copia para o Drive sem sobrescrever, remonta o Drive, confere os SHA-256 e grava o registro |

Abra o Colab logada na conta do mestrado, sem GPU (a RAM padrão deve bastar; se acabar, use o ambiente de RAM alta). Se alguma célula der erro, pare e guarde a saída inteira. A saída vai para `resultados/semana9/rq4_medidas` (ou `_2`, `_3`, se a pasta já tiver conteúdo).

Os mapas usam blocos de 10 × 10 pixels (100 m). A margem dos pixels pontuados em alguma distância entra exata; a dos demais pixels vem da margem arredondada ao quarto, usada só nos mapas.

Arquivos a enviar depois da célula 5, da pasta de saída: `medidas_por_regime.csv`, `medidas_por_execucao.csv`, `contagens_celulas.csv`, `info_medidas_rq4.json`, `medidas_sha256.csv` e as três figuras `mapas_<tile>.png`.

In [ ]:
# Célula 1 de 5: Drive, repositório e registros
import os, re, json, time, hashlib, shutil, subprocess, sys, csv
import numpy as np

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

MARGENS = f'{RAIZ}/resultados/semana9/rq4_margens'
CONFIRMATORIA = f'{RAIZ}/resultados/semana6/analise'
SAIDA_BASE = f'{RAIZ}/resultados/semana9/rq4_medidas'
SAIDA_DRIVE, n = SAIDA_BASE, 1
while os.path.isdir(SAIDA_DRIVE) and os.listdir(SAIDA_DRIVE):      # nunca sobrescreve
    n += 1
    SAIDA_DRIVE = f'{SAIDA_BASE}_{n}'
print('Saída desta rodada:', SAIDA_DRIVE.replace(MEUDRIVE, 'MyDrive'))
LOCAL = '/content/rq4_medidas'
ENTRADAS_LOCAL = f'{LOCAL}/entradas'
SAIDA_LOCAL = f'{LOCAL}/saida'
for d in (ENTRADAS_LOCAL, SAIDA_LOCAL):
    os.makedirs(d, exist_ok=True)

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
sys.path.insert(0, f'{REPO_GIT}/analysis/rq4')
sys.path.insert(0, f'{REPO_GIT}/analysis')
import nucleo as N
import mascaras as M
import inferencia as I
import medidas as D

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

def esperar(caminho, limite=180):
    t0 = time.time()
    while not os.path.exists(caminho):
        if time.time() - t0 > limite:
            raise FileNotFoundError(f'{caminho} não apareceu no Drive em {limite} s. Pare e guarde esta saída.')
        time.sleep(5)

def trazer(no_drive, sha_esperado=None, pasta=ENTRADAS_LOCAL):
    # Copia do Drive para o disco local. Com SHA-256 esperado, confere; se não conferir, remonta o Drive uma vez
    # e copia de novo (o Drive às vezes mostra uma versão velha do arquivo). Sem SHA esperado, só calcula e devolve.
    local = f'{pasta}/{os.path.basename(no_drive)}'
    for tentativa in (1, 2):
        esperar(no_drive)
        shutil.copyfile(no_drive, local + '.parcial')
        os.replace(local + '.parcial', local)
        sha = sha256_arquivo(local)
        if sha_esperado is None or sha == sha_esperado:
            return local, sha
        if tentativa == 1:
            print(f'   {os.path.basename(no_drive)}: SHA-256 diferente do registro; remontando o Drive e copiando de novo.')
            drive.flush_and_unmount()
            time.sleep(10)
            drive.mount('/content/drive', force_remount=True)
    raise ValueError(f'{no_drive}: SHA-256 {sha[:16]}..., o registro diz {sha_esperado[:16]}... Pare e guarde esta saída.')

# Registros do repositório: margens (manifesto e informações) e janelas.
MANIFESTO = {r['arquivo']: r for r in csv.DictReader(open(f'{REPO_GIT}/analysis/rq4/margens/manifesto_sha256.csv'))}
INFO = json.load(open(f'{REPO_GIT}/analysis/rq4/margens/info_inferencia_rq4.json'))
assert len(MANIFESTO) == 48, len(MANIFESTO)
assert INFO['commit_repositorio'].startswith('50f8f5c')
SHA_MODULOS = {f: hashlib.sha256(open(f'{REPO_GIT}/analysis/{f}', 'rb').read()).hexdigest()
               for f in ('rq4/mascaras.py', 'rq4/inferencia.py', 'rq4/medidas.py', 'nucleo.py')}
assert SHA_MODULOS['rq4/mascaras.py'] == INFO['modulos_sha256']['mascaras.py']
assert SHA_MODULOS['rq4/inferencia.py'] == INFO['modulos_sha256']['inferencia.py']
JANELAS_LINHAS = list(csv.DictReader(open(f'{REPO_GIT}/analysis/rq4/mascaras/rq4_janelas.csv', encoding='utf-8')))
assert len(JANELAS_LINHAS) == 4356, len(JANELAS_LINHAS)

# Análise confirmatória: τ de cada execução e valores do alvo.
res_local, SHA_RESULTADOS = trazer(f'{CONFIRMATORIA}/resultados.json')
reg_local, SHA_REGIMES = trazer(f'{CONFIRMATORIA}/tabelas/regimes.csv')
RES_C = json.load(open(res_local, encoding='utf-8'))
REGIMES_CSV = list(csv.DictReader(open(reg_local, encoding='utf-8')))
CHAVE = RES_C['meta']['chave']
assert CHAVE == INFO['codigos_semana6'], (CHAVE, INFO['codigos_semana6'])
print('Commit do repositório:', commit[:7])
print('Chave da rodada confirmatória confere com a da inferência:', CHAVE)
print('resultados.json:', SHA_RESULTADOS[:16], ' regimes.csv:', SHA_REGIMES[:16])

In [ ]:
# Célula 2 de 5: rótulos dos três tiles e τ das 15 execuções
REGIMES = ['R1', 'R2', 'R3']
SEMENTES = [1, 2, 3, 4, 5]
TILES = list(D.TILES)
TAU = {}
por_cod = {(e['codigo'], int(e['semente'])): float(e['tau']) for e in RES_C['execucoes']}
for reg in REGIMES:
    for sem in SEMENTES:
        TAU[(reg, sem)] = por_cod[(CHAVE[reg], sem)]
assert len(por_cod) == 15 and len(TAU) == 15
print('τ por execução (mínimo e máximo):', f'{min(TAU.values()):.4f} a {max(TAU.values()):.4f}')

ROT = {}
for tile in TILES:
    nome = f'rotulos_{tile}.npz'
    local, _ = trazer(f'{MARGENS}/{nome}', MANIFESTO[nome]['sha256'])
    z = np.load(local)
    ROT[tile] = {k: z[k] for k in z.files}
    os.remove(local)
    r = ROT[tile]
    assert [int(d) for d in r['distancias']] == [1000, 500, 2000]
    assert int(r['tamanhos'].sum()) == r['codigo'].shape[1] == r['pontuado'].size
    print(f"{tile}: {len(r['janelas'])} janelas, {int(r['tamanhos'].sum()):,} pixels atribuídos, "
          f"{int(r['pontuado'].sum()):,} pontuados em alguma distância")
assert [len(ROT[t]['janelas']) for t in TILES] == [INFO['tiles'][t]['janelas'] for t in TILES]
assert [int(ROT[t]['pontuado'].sum()) for t in TILES] == [INFO['tiles'][t]['pixels_pontuados_em_alguma_distancia'] for t in TILES]
print('Rótulos conferidos com o registro da inferência.')

In [ ]:
# Célula 3 de 5: medidas
from datetime import datetime, timezone
import scipy, matplotlib

T0 = time.time()
def obter_margem(regime, semente, tile):
    nome = f'run_{regime}_s{semente}_{tile}.npz'
    local, _ = trazer(f'{MARGENS}/{nome}', MANIFESTO[nome]['sha256'])
    z = np.load(local)
    m = z['margem']
    os.remove(local)
    return m

bins = N.Bins()
for volta in range(1, 5):
    res = D.calcular(obter_margem, TAU, TILES, ROT, JANELAS_LINHAS, REGIMES, SEMENTES, bins,
                     aviso=lambda s: print(s))
    print(f'Volta {volta}: bins {bins}, maior diferença entre o AUROC_d do histograma e o exato: {res["dif_hist"]:.2e}; '
          f'maior |m|: {res["max_abs_m"]:.2f}; {(time.time() - T0) / 60:.1f} min')
    if res['dif_hist'] <= N.HIST_TOL:
        break
    bins = N.refine_bins(bins, res['max_abs_m'])
else:
    raise ValueError('O AUROC_d do histograma não chegou a 1e-4 do valor exato em 4 voltas. Pare e guarde esta saída.')

DISTANCIAS = [1000, 500, 2000]
EXEC = D.tabela_execucoes(res['contagens'], TILES, REGIMES, SEMENTES, DISTANCIAS)
ALVO = D.ler_alvo(REGIMES_CSV)
eventos_alvo = sorted({e for v in ALVO.values() for e in v['eventos']})
print('Eventos do alvo lidos de regimes.csv:', eventos_alvo)
assert len(eventos_alvo) == 5, eventos_alvo
REG = D.tabela_regimes(EXEC, ALVO)
CEL = D.tabela_celulas(res['contagens'], TILES, REGIMES, SEMENTES, DISTANCIAS)
D.escrever_csv(f'{SAIDA_LOCAL}/medidas_por_execucao.csv', EXEC)
D.escrever_csv(f'{SAIDA_LOCAL}/medidas_por_regime.csv', REG)
D.escrever_csv(f'{SAIDA_LOCAL}/contagens_celulas.csv', CEL)

# Resumo na tela: todos os pixels, distância principal, os três tiles juntos e cada tile.
import pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
df = pd.DataFrame(REG)
sel = df[(df.distancia_m == 1000) & (df.tipo_grupo == 'todos')]
for med in D.MEDIDAS:
    t = sel[sel.medida == med].pivot(index='escopo', columns='regime', values='media_sementes').reindex(TILES + [D.AGREGADO])
    print(f'\n{med} (média das 5 sementes, 1 km, todos os pixels)')
    print(t.round(4).to_string())
    a = sel[(sel.medida == med) & (sel.escopo == D.AGREGADO)].set_index('regime')
    if (a.alvo_agregado != '').all():
        print('alvo agregado da análise confirmatória: ' + ', '.join(f"{r} {a.loc[r, 'alvo_agregado']:.4f}" for r in REGIMES))
        print('os 3 tiles em relação à faixa dos cinco eventos do alvo: ' + ', '.join(f"{r} {a.loc[r, 'posicao_nos_eventos']}" for r in REGIMES))
print('\nPixels e discordâncias por grupo (R1, semente 1, 1 km):')
print(pd.DataFrame(EXEC).query("regime == 'R1' and semente == 1 and distancia_m == 1000")[['escopo', 'tipo_grupo', 'grupo', 'pixels', 'discordancias']].to_string(index=False))

In [ ]:
# Célula 4 de 5: mapas
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display

COLUNAS = [('msp', 'MSP médio', 'viridis', 0.5, 1.0),
           ('sinalizado', 'Fração sinalizada (MSP < τ)', 'Reds', 0.0, 1.0),
           ('discordancia', 'Fração em discordância (pixels pontuados, 1 km)', 'Reds', 0.0, 1.0)]
SEMENTE = D.SEMENTE_DOS_MAPAS
FIGURAS = []
for tile in TILES:
    fig, eixos = plt.subplots(len(REGIMES), len(COLUNAS), figsize=(15, 4.6 * len(REGIMES)))
    for i, reg in enumerate(REGIMES):
        nome = f'run_{reg}_s{SEMENTE}_{tile}.npz'
        local, _ = trazer(f'{MARGENS}/{nome}', MANIFESTO[nome]['sha256'])
        z = np.load(local)
        b = D.blocos_do_tile(z['margem_q'], z['margem'], ROT[tile], TAU[(reg, SEMENTE)])
        os.remove(local)
        for j, (chave, titulo, cmap, lo, hi) in enumerate(COLUNAS):
            ax = eixos[i, j]
            cm = plt.get_cmap(cmap).copy(); cm.set_bad('white')
            im = ax.imshow(np.ma.masked_invalid(b[chave]), cmap=cm, vmin=lo, vmax=hi, interpolation='nearest')
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_title(f'{tile} {reg} semente {SEMENTE}: {titulo}', fontsize=9)
            fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    fig.tight_layout()
    caminho = f'{SAIDA_LOCAL}/mapas_{tile}.png'
    fig.savefig(caminho, dpi=90)
    plt.close(fig)
    FIGURAS.append(caminho)
    display(Image(caminho))
print('Mapas gravados. Branco: pixels sem saída do modelo (janela que não entra) ou, na terceira coluna, sem pontuação.')

In [ ]:
# Célula 5 de 5: Drive, conferência e registro
os.makedirs(SAIDA_DRIVE, exist_ok=True)
ARQUIVOS = sorted(f for f in os.listdir(SAIDA_LOCAL) if f.endswith(('.csv', '.png')))
assert len(ARQUIVOS) == 6, ARQUIVOS

# Registro da rodada
info = {
    'plano': 'prereg/plano_rq4.md, seção 3',
    'commit_repositorio': commit,
    'modulos_sha256': SHA_MODULOS,
    'margens': 'resultados/semana9/rq4_margens, SHA-256 conferidos com analysis/rq4/margens/manifesto_sha256.csv',
    'confirmatoria': {'resultados.json': SHA_RESULTADOS, 'regimes.csv': SHA_REGIMES, 'chave': CHAVE},
    'tau': {f'{r}_s{s}': TAU[(r, s)] for r in REGIMES for s in SEMENTES},
    'distancias_m': DISTANCIAS, 'distancia_principal_m': D.DISTANCIA_PRIMARIA,
    'bins': {'largura': bins.width, 'maximo': bins.vmax, 'n': bins.n},
    'maior_diferenca_auroc_histograma_exato': res['dif_hist'], 'maior_abs_m': res['max_abs_m'],
    'eventos_do_alvo': eventos_alvo, 'semente_dos_mapas': SEMENTE, 'bloco_dos_mapas_pixels': D.BLOCO_MAPA,
    'versoes': {'numpy': np.__version__, 'scipy': scipy.__version__, 'matplotlib': matplotlib.__version__,
                'python': sys.version.split()[0]},
    'minutos': round((time.time() - T0) / 60, 1),
    'gerado_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
}
with open(f'{SAIDA_LOCAL}/info_medidas_rq4.json', 'w', encoding='utf-8') as f:
    json.dump(info, f, indent=1, ensure_ascii=False)
ARQUIVOS = sorted(os.listdir(SAIDA_LOCAL))
SHAS = {a: sha256_arquivo(f'{SAIDA_LOCAL}/{a}') for a in ARQUIVOS}
with open(f'{SAIDA_LOCAL}/medidas_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['arquivo', 'bytes', 'sha256'])
    for a in ARQUIVOS:
        w.writerow([a, os.path.getsize(f'{SAIDA_LOCAL}/{a}'), SHAS[a]])
ARQUIVOS = sorted(os.listdir(SAIDA_LOCAL))

# Copia para o Drive como arquivo novo (nome parcial, depois renomeia); a pasta é nova, nada é sobrescrito.
for a in ARQUIVOS:
    destino = f'{SAIDA_DRIVE}/{a}'
    assert not os.path.exists(destino), f'{destino} já existe; nada é sobrescrito. Pare e guarde esta saída.'
    shutil.copyfile(f'{SAIDA_LOCAL}/{a}', f'{SAIDA_DRIVE}/parcial_{a}')
    os.rename(f'{SAIDA_DRIVE}/parcial_{a}', destino)
drive.flush_and_unmount()
time.sleep(10)
drive.mount('/content/drive', force_remount=True)
conferidos = 0
for a in ARQUIVOS:
    if a == 'medidas_sha256.csv':
        continue
    esperar(f'{SAIDA_DRIVE}/{a}')
    sha = sha256_arquivo(f'{SAIDA_DRIVE}/{a}')
    assert sha == SHAS[a], f'{a}: SHA-256 no Drive {sha[:16]}..., local {SHAS[a][:16]}... Pare e guarde esta saída.'
    conferidos += 1
print(f'{conferidos} arquivos conferidos no Drive depois de remontar, em {SAIDA_DRIVE.replace(MEUDRIVE, "MyDrive")}:')
for a in ARQUIVOS:
    print('  ', a, os.path.getsize(f'{SAIDA_DRIVE}/{a}'))
print('Me envie medidas_por_regime.csv, medidas_por_execucao.csv, contagens_celulas.csv, info_medidas_rq4.json, medidas_sha256.csv e as três figuras.')